In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import GroupShuffleSplit

from src.utils.paths import load_paths
from src.utils.logging import setup_logger

from src.features.extract import load_feature_config, extract_features_from_flows, feature_config_hash_text
from src.pipeline.feature_pipeline import FeaturePipeline
from src.pipeline.artifacts import default_feature_artifacts

paths = load_paths()
paths.ensure_dirs()
logger = setup_logger(level="INFO")

logger.info(f"Repo root: {paths.repo_root}")
logger.info(f"Processed dir: {paths.data_processed}")
logger.info(f"Configs dir: {paths.configs_dir}")
logger.info(f"Artifacts dir: {paths.artifacts_dir}")


2026-03-04 07:00:08 | INFO | ai-vpn-firewall | Repo root: C:\Users\scoti\PycharmProjects\ai-vpn-firewall
2026-03-04 07:00:08 | INFO | ai-vpn-firewall | Processed dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed
2026-03-04 07:00:08 | INFO | ai-vpn-firewall | Configs dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\configs
2026-03-04 07:00:08 | INFO | ai-vpn-firewall | Artifacts dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts


In [2]:
features_yaml = paths.configs_dir / "features.yaml"
cfg = load_feature_config(features_yaml)

logger.info("Loading flows...")
vnat_flows = pd.read_parquet(paths.data_processed / "vnat" / "flows.parquet")
iscx_flows = pd.read_parquet(paths.data_processed / "iscx" / "flows.parquet")

logger.info("Extracting features from flows (VNAT)...")
vnat_feats = extract_features_from_flows(vnat_flows, cfg)
vnat_feats["dataset"] = "vnat"

logger.info("Extracting features from flows (ISCX)...")
iscx_feats = extract_features_from_flows(iscx_flows, cfg)
iscx_feats["dataset"] = "iscx"

# Bring split from old files (your existing approach)
vnat_old = pd.read_parquet(paths.data_processed / "vnat" / "features_trainable.parquet")
split_map_vnat = dict(zip(vnat_old["capture_id"], vnat_old["split"]))
vnat_feats["split"] = vnat_feats["capture_id"].map(split_map_vnat).fillna("unknown")

iscx_old = pd.read_parquet(paths.data_processed / "iscx" / "features.parquet")
split_map_iscx = dict(zip(iscx_old["capture_id"], iscx_old["split"]))
iscx_feats["split"] = iscx_feats["capture_id"].map(split_map_iscx).fillna("unknown")

# Drop unknown + apply min packets filter (your approach)
vnat_feats = vnat_feats[vnat_feats["split"] != "unknown"].copy()
iscx_feats = iscx_feats[iscx_feats["split"] != "unknown"].copy()

if "q_min_packets_ok" in vnat_feats.columns:
    vnat_feats = vnat_feats[vnat_feats["q_min_packets_ok"] == 1.0].copy()
if "q_min_packets_ok" in iscx_feats.columns:
    iscx_feats = iscx_feats[iscx_feats["q_min_packets_ok"] == 1.0].copy()

df_all = pd.concat([vnat_feats, iscx_feats], ignore_index=True)

# Normalize ISCX split naming (your approach)
df_all["split"] = df_all["split"].replace({"iscx_train": "train", "iscx_val": "val", "iscx_test": "test"})

logger.info(f"Combined df_all: {df_all.shape}, columns={len(df_all.columns)}")
df_all["split"].value_counts()


2026-03-04 07:00:08 | INFO | ai-vpn-firewall | Loading flows...
2026-03-04 07:00:09 | INFO | ai-vpn-firewall | Extracting features from flows (VNAT)...
2026-03-04 07:00:49 | INFO | ai-vpn-firewall | Extracting features from flows (ISCX)...
2026-03-04 07:01:50 | INFO | ai-vpn-firewall | Combined df_all: (19908, 48), columns=48


split
train    16138
val       1926
test      1844
Name: count, dtype: int64

In [3]:
logger.info("Rebalancing VNAT splits...")
vnat_mask = df_all["dataset"] == "vnat"
vnat_df = df_all[vnat_mask].copy()

gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=42)
train_idx, temp_idx = next(gss.split(vnat_df, groups=vnat_df["capture_id"]))
vnat_train = vnat_df.iloc[train_idx].copy()
vnat_temp  = vnat_df.iloc[temp_idx].copy()

gss_val_test = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val_test.split(vnat_temp, groups=vnat_temp["capture_id"]))
vnat_val  = vnat_temp.iloc[val_idx].copy()
vnat_test = vnat_temp.iloc[test_idx].copy()

vnat_train = vnat_train.copy(); vnat_train["split"] = "train"
vnat_val   = vnat_val.copy();   vnat_val["split"]   = "val"
vnat_test  = vnat_test.copy();  vnat_test["split"]  = "test"

vnat_balanced = pd.concat([vnat_train, vnat_val, vnat_test], ignore_index=True)
df_all = pd.concat([df_all[~vnat_mask], vnat_balanced], ignore_index=True)

# Enforce Voipbuster split (match your real IDs)
vb_train_cap = "vpn_vpn_voipbuster1a.pcap"
vb_test_cap  = "vpn_vpn_voipbuster1b.pcap"

df_all.loc[df_all["capture_id"] == vb_train_cap, "split"] = "train"
df_all.loc[df_all["capture_id"] == vb_test_cap,  "split"] = "test"

print("Final splits:")
print(df_all["split"].value_counts())


2026-03-04 07:01:50 | INFO | ai-vpn-firewall | Rebalancing VNAT splits...
Final splits:
split
train    14260
val       3217
test      2431
Name: count, dtype: int64


In [4]:
# --- A) Label Sanity Check ---
def label_sanity(flows, name):
    print("\n" + "="*60)
    print(f"LABEL SANITY: {name}")
    print("="*60)

    # Basic distribution
    print("Flow-level label distribution:")
    print(flows["label"].value_counts(normalize=True))

    # Compare capture name prefix vs label mean
    flows = flows.copy()
    flows["is_vpn_prefix"] = flows["capture_id"].astype(str).str.startswith("vpn_").astype(int)

    by_cap = flows.groupby("capture_id").agg(
        n=("label", "size"),
        label_mean=("label", "mean"),
        prefix=("is_vpn_prefix", "max")
    ).reset_index()

    print("\nCaptures where prefix says VPN but labels are mostly 0:")
    print(by_cap[(by_cap["prefix"]==1) & (by_cap["label_mean"] < 0.5)].head(20))

    print("\nCaptures where prefix says NONVPN but labels are mostly 1:")
    print(by_cap[(by_cap["prefix"]==0) & (by_cap["label_mean"] > 0.5)].head(20))

label_sanity(vnat_flows, "VNAT")
label_sanity(iscx_flows, "ISCX")



LABEL SANITY: VNAT
Flow-level label distribution:
label
0    0.988757
1    0.011243
Name: proportion, dtype: float64

Captures where prefix says VPN but labels are mostly 0:
Empty DataFrame
Columns: [capture_id, n, label_mean, prefix]
Index: []

Captures where prefix says NONVPN but labels are mostly 1:
Empty DataFrame
Columns: [capture_id, n, label_mean, prefix]
Index: []

LABEL SANITY: ISCX
Flow-level label distribution:
label
0    0.707473
1    0.292527
Name: proportion, dtype: float64

Captures where prefix says VPN but labels are mostly 0:
Empty DataFrame
Columns: [capture_id, n, label_mean, prefix]
Index: []

Captures where prefix says NONVPN but labels are mostly 1:
Empty DataFrame
Columns: [capture_id, n, label_mean, prefix]
Index: []


In [5]:
# --- B) Feature Shift Check ---
def compare_feature_stats(df_all, feat, label=1):
    sub_v = df_all[(df_all["dataset"]=="vnat") & (df_all["label"]==label)]
    sub_i = df_all[(df_all["dataset"]=="iscx") & (df_all["label"]==label)]

    print("\n" + "="*60)
    print(f"FEATURE SHIFT CHECK: {feat} (label={label})")
    print("="*60)

    for name, sub in [("VNAT", sub_v), ("ISCX", sub_i)]:
        if sub.empty or feat not in sub.columns:
            print(f"{name}: missing")
            continue
        print(f"{name}: n={len(sub)} mean={sub[feat].mean():.4f} std={sub[feat].std():.4f} "
              f"p25={sub[feat].quantile(0.25):.4f} p50={sub[feat].median():.4f} p75={sub[feat].quantile(0.75):.4f}")

# Check key features that might flip with direction
for f in ["f_pkt_imbalance", "f_byte_imbalance", "f_iat_burstiness"]:
    compare_feature_stats(df_all, f, label=1)
    compare_feature_stats(df_all, f, label=0)



FEATURE SHIFT CHECK: f_pkt_imbalance (label=1)
VNAT: n=374 mean=0.2040 std=0.3891 p25=0.0000 p50=0.0000 p75=0.0000
ISCX: n=2943 mean=0.6463 std=0.3510 p25=0.5000 p50=0.7500 p75=0.9608

FEATURE SHIFT CHECK: f_pkt_imbalance (label=0)
VNAT: n=7733 mean=0.8971 std=0.1556 p25=0.8108 p50=1.0000 p75=1.0000
ISCX: n=8858 mean=0.5189 std=0.3964 p25=0.0000 p50=0.6000 p75=0.9000

FEATURE SHIFT CHECK: f_byte_imbalance (label=1)
VNAT: n=374 mean=0.1838 std=0.3654 p25=0.0000 p50=0.0000 p75=0.0000
ISCX: n=2943 mean=0.3932 std=0.3316 p25=0.0468 p50=0.3659 p75=0.6926

FEATURE SHIFT CHECK: f_byte_imbalance (label=0)
VNAT: n=7733 mean=0.6985 std=0.2798 p25=0.4995 p50=0.8730 p75=0.8730
ISCX: n=8858 mean=0.3656 std=0.3373 p25=0.0000 p50=0.3551 p75=0.6588

FEATURE SHIFT CHECK: f_iat_burstiness (label=1)
VNAT: n=374 mean=0.9071 std=1.3780 p25=0.3333 p50=0.3334 p75=0.3336
ISCX: n=2943 mean=2.3016 std=1.5900 p25=1.1111 p50=1.6954 p75=3.3652

FEATURE SHIFT CHECK: f_iat_burstiness (label=0)
VNAT: n=7733 mean=2.3

In [6]:
# --- C) Cross-Domain Sign Flip Diagnostic ---
def threshold_sweep(y, p, tag=""):
    print(f"\n--- Threshold Sweep: {tag} ---")
    for thr in [0.1, 0.3, 0.5, 0.7, 0.9]:
        acc = accuracy_score(y, (p >= thr).astype(int))
        acc_inv = accuracy_score(y, ((1-p) >= thr).astype(int))
        print(f"{tag} thr={thr:.1f}: acc(p)={acc:.3f} acc(1-p)={acc_inv:.3f}")

def smoke_cross_domain_sweep(df_all):
    print("\n=== C) Cross-domain Sign Flip Diagnostic ===")

    df_vnat_train = df_all[(df_all["dataset"] == "vnat") & (df_all["split"] == "train")].copy()
    df_iscx_test = df_all[(df_all["dataset"] == "iscx") & (df_all["split"] == "test")].copy()

    if df_vnat_train.empty or df_iscx_test.empty:
        print("Skipping cross-domain test: missing data.")
        return

    pipe_vnat = FeaturePipeline().fit(df_vnat_train)
    feat_cols = pipe_vnat.model_feature_names()

    X_vnat = pipe_vnat.transform(df_vnat_train)
    X_iscx = pipe_vnat.transform(df_iscx_test)

    y_vnat = df_vnat_train["label"].values
    y_iscx = df_iscx_test["label"].values

    model = xgb.XGBClassifier(
        n_estimators=30,
        max_depth=3,
        learning_rate=0.1,
        n_jobs=4,
        eval_metric="auc",
        random_state=42
    )
    model.fit(X_vnat[feat_cols], y_vnat)

    preds = model.predict_proba(X_iscx[feat_cols])[:, 1]

    auc = roc_auc_score(y_iscx, preds)
    print(f"VNAT->ISCX AUC: {auc:.4f}")

    threshold_sweep(y_iscx, preds, tag="VNAT->ISCX")

smoke_cross_domain_sweep(df_all)



=== C) Cross-domain Sign Flip Diagnostic ===
VNAT->ISCX AUC: 0.1437

--- Threshold Sweep: VNAT->ISCX ---
VNAT->ISCX thr=0.1: acc(p)=0.625 acc(1-p)=0.145
VNAT->ISCX thr=0.3: acc(p)=0.669 acc(1-p)=0.276
VNAT->ISCX thr=0.5: acc(p)=0.694 acc(1-p)=0.306
VNAT->ISCX thr=0.7: acc(p)=0.724 acc(1-p)=0.331
VNAT->ISCX thr=0.9: acc(p)=0.855 acc(1-p)=0.375


In [7]:
# --- D) Refined Leave-One-App-Out (LOAO) ---
print("\n=== D) Refined Leave-One-App-Out (LOAO) ===")

# 1. Map capture_id -> app
def make_capture_to_app_map(flows: pd.DataFrame) -> pd.Series:
    if "app" not in flows.columns:
        return pd.Series(dtype=str)
    def mode_or_unknown(x):
        x = x.dropna()
        if len(x) == 0: return "unknown"
        return x.value_counts().idxmax()
    return flows.groupby("capture_id")["app"].apply(mode_or_unknown)

cap2app_vnat = make_capture_to_app_map(vnat_flows)
cap2app_iscx = make_capture_to_app_map(iscx_flows)
cap2app = pd.concat([cap2app_vnat, cap2app_iscx])

df_all["app"] = df_all["capture_id"].map(cap2app).fillna("unknown")

# 2. Normalize app names
df_all["app"] = (
    df_all["app"].astype(str)
    .str.lower()
    .str.strip()
    .replace({
        "skype_video": "skype",
        "skype_audio": "skype",
        "facebook_audio": "facebook",
        "facebook_chat": "facebook",
        "aim_chat": "aim",
        "icq_chat": "icq",
        "gmail_chat": "gmail",
        "hangouts_chat": "hangouts",
    }, regex=True)
)
# Manual overrides for known patterns
df_all.loc[df_all["app"].str.contains("skype"), "app"] = "skype"
df_all.loc[df_all["app"].str.contains("facebook"), "app"] = "facebook"
df_all.loc[df_all["app"].str.contains("netflix"), "app"] = "netflix"
df_all.loc[df_all["app"].str.contains("youtube"), "app"] = "youtube"
df_all.loc[df_all["app"].str.contains("spotify"), "app"] = "spotify"
df_all.loc[df_all["app"].str.contains("vimeo"), "app"] = "vimeo"
df_all.loc[df_all["app"].str.contains("bittorrent"), "app"] = "bittorrent"

# 3. Run LOAO on TRAIN split only
df_train_loao = df_all[df_all["split"] == "train"].copy()
apps = sorted([a for a in df_train_loao["app"].unique() if a != "unknown"])

MIN_N = 200
MIN_POS = 20

loao_results = []
print(f"Testing {len(apps)} apps: {apps}")

for holdout_app in apps:
    # Hold out ALL captures for this app
    train_df = df_train_loao[df_train_loao["app"] != holdout_app].copy()
    test_df  = df_train_loao[df_train_loao["app"] == holdout_app].copy()

    # Guardrails
    if len(test_df) < MIN_N:
        # print(f"Skipping {holdout_app}: n={len(test_df)} < {MIN_N}")
        continue

    n_pos = test_df["label"].sum()
    n_neg = (test_df["label"] == 0).sum()

    if n_pos < MIN_POS or n_neg < MIN_POS:
        # print(f"Skipping {holdout_app}: unbalanced (pos={n_pos}, neg={n_neg})")
        continue

    pipe_loao = FeaturePipeline().fit(train_df)
    feats_loao = pipe_loao.model_feature_names()

    X_tr = pipe_loao.transform(train_df)[feats_loao]
    y_tr = train_df["label"].astype(int)

    X_te = pipe_loao.transform(test_df)[feats_loao]
    y_te = test_df["label"].astype(int)

    model_loao = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        random_state=42,
        n_jobs=4,
    )
    model_loao.fit(X_tr, y_tr)

    p = model_loao.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, p)

    loao_results.append({
        "app": holdout_app,
        "n_test": len(test_df),
        "pos_rate": float(y_te.mean()),
        "auc": float(auc)
    })
    print(f"LOAO app={holdout_app:>15s} | n={len(test_df):4d} | AUC={auc:.4f}")

if loao_results:
    res_df = pd.DataFrame(loao_results).sort_values("auc")
    print("\nLOAO Summary (Worst 10):")
    print(res_df.head(10).to_string(index=False))
    print(f"\nMacro Mean AUC: {res_df['auc'].mean():.4f}")
else:
    print("No valid LOAO runs passed the filters.")



=== D) Refined Leave-One-App-Out (LOAO) ===
Testing 48 apps: ['aim', 'email2a', 'facebook', 'ftps', 'hangouts', 'icq', 'netflix', 'nonaim', 'nonaimchat1', 'nonaimchat2', 'nonemail1a', 'nonemail1b', 'nonemail2a', 'nonemail2b', 'nonftps', 'nongmailchat1', 'nongmailchat2', 'nongmailchat3', 'nonhangout', 'nonhangouts', 'nonicq', 'nonscp1', 'nonscpdown1', 'nonscpdown2', 'nonscpdown3', 'nonscpdown4', 'nonscpdown5', 'nonscpdown6', 'nonscpup1', 'nonscpup3', 'nonscpup5', 'nonsftp', 'nonsftpdown1', 'nonsftpup1', 'nonvoipbuster', 'nonvoipbuster1b', 'nonvoipbuster2b', 'rdp', 'rsync', 'scp', 'sftp', 'skype', 'spotify', 'ssh', 'vimeo', 'voip', 'voipbuster1a', 'youtube']
LOAO app=       facebook | n=1413 | AUC=0.9764
LOAO app=        netflix | n= 448 | AUC=0.9921
LOAO app=          skype | n=2796 | AUC=0.8963
LOAO app=          vimeo | n= 319 | AUC=0.9994
LOAO app=        youtube | n= 620 | AUC=0.9103

LOAO Summary (Worst 10):
     app  n_test  pos_rate      auc
   skype    2796  0.282546 0.896321
 